***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import ipywidgets as widgets

pd.options.display.float_format = '{:.1f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_out  = os.path.join(path_users
                             , 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents'
                             , 'Process Revamp'
                             , 'Task 9. Collect new data'
                             , 'Census')

path_code    = os.path.join(path_git, 'Data', 'EIA')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://www.eia.gov/opendata/documentation.php
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

***

Route 1

***

In [ ]:
root_ = 'https://api.eia.gov/v2'
api_key_ = f'/?api_key={api_key}'


has_routes = ['coal', 'electricity', 'natural-gas', 'nuclear-outages', 'petroleum', 'densified-biomass', 'aeo', 'co2-emissions']
categories_to_keep = ['coal', 'electricity', 'natural-gas', 'nuclear-outages', 'petroleum', 'densified-biomass', 'co2-emissions']

list_df_cat = []

for cat in tqdm(categories_to_keep):

    cat_ = f'/{cat}'
    
    query = f"{root_}{cat_}{api_key_}"
    
    # Use requests package to call out to the API
    response = requests.get(query).text
    response = response.replace('null', '"null"')
    response = ast.literal_eval(response)
    
    
    list_df_id = []

    if cat in has_routes:
        for id in range(len(response['response']['routes'])):
            df_routes = pd.DataFrame(response['response']['routes'][id], index = [0])
            df_routes = df_routes.rename(columns = {'id':'route', 'name':'route_name'})
            list_df_id.append(df_routes)
    
    
    df_routes = pd.concat(list_df_id)
    df_routes = df_routes.reset_index(drop = True)
    
    
    df_routes['category'] = cat
    list_df_cat.append(df_routes)


df_routes = pd.concat(list_df_cat)
df_routes = df_routes.reset_index(drop = True)
df_routes = df_routes.set_index('category').reset_index()
df_routes['route_name'] = df_routes['route_name'].str.replace('\\/', '/')
df_routes['route_name'] = df_routes['route_name'].str.replace('\\' , '/')
df_routes['route_name'] = df_routes['route_name'].str.replace(' / ', '/')

display(df_routes.head(), df_routes.tail())

***

Route 2

***

In [ ]:
categories = list(df_routes.category.unique())
categories

In [ ]:
root_ = 'https://api.eia.gov/v2'


list_df_cat = []

for cat in categories:

    print(cat)
    cat_ = f'/{cat}'
    
    routes = list(df_routes[df_routes['category'] == cat].route1.unique())
    print(routes)
    
    list_df_routes = []
    
    for route in routes:
    
        try:
            route_ = f'/{route}'
            api_key_ = f'/?api_key={api_key}'
            
            query = f"{root_}{cat_}{route_}{api_key_}"
            
            print(query)    
            
            # Use requests package to call out to the API
            response = requests.get(query).text
            response = response.replace('null', '"null"')
            response = ast.literal_eval(response)
            
            df_route = pd.DataFrame(response['response']['routes'])
            df_route.columns = ['route2', 'route2_name', 'route2_description']
            df_route['category'] = cat
            df_route['route1'  ] = route
            list_df_routes.append(df_route)
        except:
            pass

    try:
        df_cat = pd.concat(list_df_routes)
        df_cat = df_cat.reset_index(drop=True)
        df_cat = df_cat.set_index(['category', 'route1']).reset_index()
        list_df_cat.append(df_cat)
    except:
        pass
    print('')

df_routes2 = pd.concat(list_df_cat)

display(df_routes2.head(), df_routes2.tail())

In [ ]:
df_routes = df_routes.merge(df_routes2, how = 'left', on = ['category', 'route1'])
df_routes

***

Facets

***

In [ ]:
categories = df_routes[~df_routes['route2'].isna()]
categories = categories['category'].unique()
categories

In [ ]:
root_ = 'https://api.eia.gov/v2'



list_df_cats = []
for cat in categories:
    print('')
    print(cat)
    print('')
    cat_ = f'/{cat}'
    
    routes1 = df_routes[df_routes['category'] == cat]['route1'].unique()
    print(routes1)
    
    list_df_routes = []
    
    for route1 in routes1:
        print('')
        print(route1)
        routes2 = df_routes[df_routes['route1'] == route1]['route2'].unique()
        print(routes2)

        list_df_facets = []
        
        for route2 in tqdm(routes2):

            try:
                route_ = f'/{route1}/{route2}/facet'
                
                api_key_ = f'/?api_key={api_key}'
                
                query = f"{root_}{cat_}{route_}{api_key_}"
                                
                # Use requests package to call out to the API
                response = requests.get(query).text
                response = response.replace('null', '"null"')
                response = ast.literal_eval(response)
                
                facets = response['response']['facetOptions']
                df_facets = pd.DataFrame(facets, columns = ['facetOptions'])
                df_facets['category'] = cat
                df_facets['route1'] = route1
                df_facets['route2'] = route2
                list_df_facets.append(df_facets)
            except:
                pass
        try:        
            df_route_facets = pd.concat(list_df_facets)
            df_route_facets = df_route_facets.reset_index(drop = True)
            list_df_routes.append(df_route_facets)
        except:
            pass
        
    df_route_facets = pd.concat(list_df_routes)
    df_route_facets = df_route_facets.reset_index(drop = True)
    list_df_cat.append(df_route_facets)

df_cat_route_facets = pd.concat(list_df_cat)
df_cat_route_facets = df_cat_route_facets.reset_index(drop = True)
df_cat_route_facets = df_cat_route_facets.set_index(['category', 'route1', 'route2']).reset_index()
df_cat_route_facets = df_cat_route_facets.drop(['route2_name', 'route2_description'], axis = 1)
df_cat_route_facets = df_cat_route_facets.dropna()
df_cat_route_facets

In [ ]:
df_routes = df_routes.merge(df_cat_route_facets, how = 'left', on = ['category', 'route1', 'route2'])
df_routes

In [ ]:

root_ = 'https://api.eia.gov/v2'

df_loop = df_routes.copy()
df_loop = df_loop[~df_loop['route2'].isna()]

categories = df_loop['category'].unique()

list_df_cat = []

for cat in categories:
    print('')
    print('')
    print(f'Category: {cat}')
    print('')
    print('')
    df = df_loop[df_loop['category'] == cat]
    routes1 = df['route1'].unique()
    print(f'Routes available: {routes1}')
    print('')

    list_df_route1 = []

    for route1 in routes1:
        print('')
        print(f'Route1: {route1}')
        print('')
        df = df_loop[df_loop['category'] == cat]
        df = df[df['route1'] == route1]
        routes2 = df['route2'].unique()
        print(f'Routes available: {routes2}')
        
        list_df_route2 = []

        for route2 in routes2:
            print('')
            print(f'Route2: {route2}')
            df = df_loop[df_loop['category'] == cat]
            df = df[df['route1'] == route1]
            df = df[df['route2'] == route2]
            facets = df['facetOptions'].unique()
            print(f'Facets: {facets}')

            list_df_facets = []

            for facet in tqdm(facets):
                try:
                    route_ = f'/{cat}/{route1}/{route2}/facet/{facet}'
                    api_key_ = f'/?api_key={api_key}'
                    
                    query = f"{root_}{route_}{api_key_}"
                                    
                    # Use requests package to call out to the API
                    response = requests.get(query).text
                    response = response.replace('null', '"null"')
                    response = ast.literal_eval(response)
                    
                    df_facet = pd.DataFrame(response['response']['facets'])
                    df_facet['category'] = cat
                    df_facet['route1'] = route1
                    df_facet['route2'] = route2
                    df_facet['facetOption'] = facet
                    list_df_facets.append(df_facet)
                except:
                    pass
            try:
                df_facets = pd.concat(list_df_facets)
                df_facets = df_facets.reset_index(drop=True)
                list_df_route2.append(df_facets)
            except:
                pass
        try:
            df_route2 = pd.concat(list_df_route2)
            df_route2 = df_route2.reset_index(drop=True)
            list_df_route1.append(df_route2)
        except:
            pass
    try:
        df_route1 = pd.concat(list_df_route1)
        df_route1 = df_route1.reset_index(drop=True)
        list_df_cat.append(df_route1)
    except:
        pass
            
df_cat = pd.concat(list_df_cat)
df_cat = df_cat.reset_index(drop = True)
df_cat = df_cat.set_index(['category', 'route1', 'route2', 'facetOption']).reset_index()
df_cat


***

Exporting

***

In [ ]:
# df_cat.to_excel(os.path.join(path_config, 'All Routes and Facets FINAL.xlsx'), index=False)